## Installing Semantic Link Labs

In [1]:
%pip install semantic-link-labs

StatementMeta(, fbb68ce2-4cd5-4055-9972-0dedae990885, 7, Finished, Available, Finished)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.9/47.9 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 859.3/859.3 kB 8.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 18.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.1/45.1 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.2/125.2 kB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 217.9/217.9 kB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.0/43.0 kB 15.5 MB/s eta 0:00:00
  Attempting uninstall: azure-core
    Found existing installation: azure-core 1.30.2
    Not uninstalling azure-core at /home/trusted-service-user/cluster-env/trident_env/lib/python3.11/site-packages, outside environment /nfs4/pyenv-6612f4d7-3f8e-47e5-a2c6-7e8f46fefe13
    Can't uninstall 'azure-core'. No files were found to uninstall.
  Attempting uninstall: semantic-link-sempy
    Found existing installation: semantic

## Importing DAXLib and Setting Package Details

In [2]:
import sempy_labs.daxlib as daxlib

package_name = 'Bacci.Time' 
dataset = 'new work' 
workspace = 'Dev Test' 

StatementMeta(, fbb68ce2-4cd5-4055-9972-0dedae990885, 9, Finished, Available, Finished)

## How to get a DAXLib Package's TMDL

In [3]:
tmdl = daxlib.get_package_tmdl(package_name=package_name, version=None) 
print(tmdl)

StatementMeta(, fbb68ce2-4cd5-4055-9972-0dedae990885, 10, Finished, Available, Finished)

/// Returns a string in the format 'n years n months n days' using month limits to give a more accurate human duration
function 'Bacci.Time.Duration' = ```
        (
                	// The first date
                	date1 : SCALAR DATETIME VAL,
                
                	// The second date
                	date2 : SCALAR DATETIME VAL
                )
                =>
             
            VAR d1 = MIN(date1, date2)
        	VAR d2 = MAX(date1, date2)
            VAR diff = DATEDIFF(d1,d2, MONTH)
            VAR series = GENERATESERIES(0,diff+0, 1)
            VAR t = FILTER(ADDCOLUMNS(series, "limit", EDATE(d1, [Value])), [limit] <= d2)
            VAR val = MAXX(t, [Value])
            VAR limit = MAXX(t, [limit])
            
            VAR y = QUOTIENT(val, 12)
            VAR yFormat = IF(y<>0, y & " year" & IF(ABS(y)=1," ","s "),BLANK()) 
            VAR m = MOD(val,12)
            VAR mFormat = IF(m<>0, m & " month" & IF(ABS(m)=1," ","s "),BLANK()) 
            V

## Extract All functions in a DAXLib Package

In [4]:
funcs = daxlib.get_package_functions(package_name=package_name, version=None) 
display(funcs)

StatementMeta(, fbb68ce2-4cd5-4055-9972-0dedae990885, 11, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 376b79c6-ecf5-4e06-a0df-d1409b506f9b)

## Adding a Package to a Semantic Model without it

In [5]:
daxlib.add_package_to_semantic_model(
    dataset=dataset, 
    package_name=package_name, 
    version=None, 
    workspace=workspace) 

StatementMeta(, fbb68ce2-4cd5-4055-9972-0dedae990885, 12, Finished, Available, Finished)

🟢 The 'new work' semantic model within the 'Dev Test' workspace has been updated to include the function(s) within the 'Bacci.Time' DAXLib.org package (version '0.1.0').


## Updating a Package in a Semantic Model

In [6]:
daxlib.update_package_in_semantic_model(dataset=dataset, package_name=package_name, version=None, workspace=workspace) 

StatementMeta(, 464eccb9-f713-4edf-909d-ccddcd6a572b, 13, Finished, Available, Finished)

🟢 The 'new work' semantic model within the 'Dev Test' workspace has been updated to include the function(s) within the 'DaxLib.FormatString' DAXLib.org package (version '0.1.1').


## Removing a Package from a Semantic Model

In [6]:
daxlib.remove_package_from_semantic_model(dataset=dataset, package_name=package_name, workspace=workspace) # Removes a package from the semantic model(Edited)

StatementMeta(, fbb68ce2-4cd5-4055-9972-0dedae990885, 13, Finished, Available, Finished)

🟢 The 'new work' semantic model within the 'Dev Test' workspace has been updated to remove the function(s) from the 'Bacci.Time' DAXLib.org package (version '0.1.0').


In [7]:
from sempy_labs.tom import connect_semantic_model
import sempy.fabric as fab 

def add_or_update(dataset, workspace, package_name, version):
    def package_exists(tom, package_name: str) -> bool:
        return any(
            f.Name.lower().startswith(f"{package_name.lower()}.")
            for f in tom.model.Functions
        )

    with connect_semantic_model(
            dataset=dataset, workspace=workspace, readonly=False
        ) as tom:
            cL = tom.model.Database.CompatibilityLevel
            if cL < 1702:
                tom.set_compatibility_level(1702)
            exists = package_exists(tom, package_name)
            if exists:
                daxlib.update_package_in_semantic_model(dataset=dataset, package_name=package_name, version=version, workspace=workspace)
            else:
                daxlib.add_package_to_semantic_model(dataset=dataset, package_name=package_name, version=version, workspace=workspace) 

            
all_sms = fab.list_datasets("Dev Test")
for dataset in all_sms['Dataset Name']:
    add_or_update(dataset=dataset, workspace=workspace,package_name=package_name,version=None)


StatementMeta(, fbb68ce2-4cd5-4055-9972-0dedae990885, 14, Finished, Available, Finished)

🟢 The 'new work' semantic model within the 'Dev Test' workspace has been updated to include the function(s) within the 'Bacci.Time' DAXLib.org package (version '0.1.0').
🟢 The 'Dataflow_Proxy_Model' semantic model within the 'Dev Test' workspace has been updated to include the function(s) within the 'Bacci.Time' DAXLib.org package (version '0.1.0').
🟢 The 'Sales Model' semantic model within the 'Dev Test' workspace has been updated to include the function(s) within the 'Bacci.Time' DAXLib.org package (version '0.1.0').
🟢 The 'Values' semantic model within the 'Dev Test' workspace has been updated to include the function(s) within the 'Bacci.Time' DAXLib.org package (version '0.1.0').
🟢 The 'new' semantic model within the 'Dev Test' workspace has been updated to include the function(s) within the 'Bacci.Time' DAXLib.org package (version '0.1.0').
